# Day 06 — Gradient Descent from Scratch

> **Goal:** Implement gradient descent manually to learn `y = wx + b` — see exactly how a model learns.

## The Algorithm

Gradient descent repeats these steps:
1. **Forward pass:** compute predictions with current weights
2. **Compute loss:** how wrong are we?
3. **Backward pass:** compute gradients (∂loss/∂weight)
4. **Update weights:** move weights in the direction that reduces loss

$$w_{\text{new}} = w_{\text{old}} - \text{lr} \times \frac{\partial \text{loss}}{\partial w}$$

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 1. Create Synthetic Data

In [ ]:
# Ground truth: y = 2x + 1
X = torch.linspace(-3, 3, 100)
y_true = 2 * X + 1 + 0.3 * torch.randn(100)  # Add noise

plt.figure(figsize=(8, 5))
plt.scatter(X.numpy(), y_true.numpy(), alpha=0.5, s=15)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Synthetic Data: y = 2x + 1 + noise")
plt.grid(True, alpha=0.3)
plt.show()

## 2. Initialize Parameters

In [ ]:
# Start with random (bad) values
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

print(f"Initial w = {w.item():.4f}, b = {b.item():.4f}")
print(f"Target:  w = 2.0000, b = 1.0000")

## 3. Training Loop — The Heart of Deep Learning

In [ ]:
learning_rate = 0.01
num_epochs = 200
losses = []
w_history = []
b_history = []

for epoch in range(num_epochs):
    # ═══ STEP 1: Forward Pass ═══
    y_pred = w * X + b
    
    # ═══ STEP 2: Compute Loss (MSE) ═══
    loss = ((y_pred - y_true) ** 2).mean()
    losses.append(loss.item())
    w_history.append(w.item())
    b_history.append(b.item())
    
    # ═══ STEP 3: Backward Pass ═══
    loss.backward()  # Fills w.grad and b.grad
    
    # ═══ STEP 4: Update Weights ═══
    with torch.no_grad():  # Don't track the update itself
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    
    # ═══ STEP 5: Zero Gradients ═══
    w.grad.zero_()
    b.grad.zero_()
    
    if (epoch + 1) % 40 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | w: {w.item():.4f} | b: {b.item():.4f}")

print(f"\nFinal: w = {w.item():.4f}, b = {b.item():.4f}")
print(f"Target: w = 2.0000, b = 1.0000")

## 4. Visualize Training

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss curve
axes[0].plot(losses, linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Loss Over Time")
axes[0].grid(True, alpha=0.3)

# Weight trajectory
axes[1].plot(w_history, label="w", linewidth=2)
axes[1].plot(b_history, label="b", linewidth=2)
axes[1].axhline(y=2.0, color="blue", linestyle="--", alpha=0.5, label="w target")
axes[1].axhline(y=1.0, color="orange", linestyle="--", alpha=0.5, label="b target")
axes[1].set_xlabel("Epoch")
axes[1].set_title("Weight Convergence")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Fitted line
with torch.no_grad():
    y_fit = w * X + b
axes[2].scatter(X.numpy(), y_true.numpy(), alpha=0.3, s=15, label="Data")
axes[2].plot(X.numpy(), y_fit.numpy(), "r-", linewidth=2, label=f"Fit: y={w.item():.2f}x+{b.item():.2f}")
axes[2].set_xlabel("x")
axes[2].set_ylabel("y")
axes[2].set_title("Learned Line vs Data")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. The Effect of Learning Rate

The learning rate controls how big each step is:
- **Too small** → training is very slow
- **Too large** → training overshoots and may diverge
- **Just right** → smooth convergence

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
lr_values = [0.0001, 0.01, 0.1]
titles = ["Too Small (lr=0.0001)", "Good (lr=0.01)", "Too Large (lr=0.1)"]

for ax, lr, title in zip(axes, lr_values, titles):
    w = torch.tensor(0.0, requires_grad=True)
    b = torch.tensor(0.0, requires_grad=True)
    losses_lr = []
    
    for epoch in range(200):
        y_pred = w * X + b
        loss = ((y_pred - y_true) ** 2).mean()
        losses_lr.append(loss.item())
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    
    ax.plot(losses_lr, linewidth=2)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, max(10, min(losses_lr[0], 50)))

plt.tight_layout()
plt.show()

## 6. Loss Landscape Visualization

Let's visualize loss as a 2D surface over w and b.

In [ ]:
# Compute loss for a grid of (w, b) values
w_range = torch.linspace(-1, 5, 100)
b_range = torch.linspace(-3, 5, 100)
W_grid, B_grid = torch.meshgrid(w_range, b_range, indexing="ij")

L_grid = torch.zeros_like(W_grid)
for i in range(100):
    for j in range(100):
        y_pred = W_grid[i, j] * X + B_grid[i, j]
        L_grid[i, j] = ((y_pred - y_true) ** 2).mean()

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
contour = ax.contourf(W_grid.numpy(), B_grid.numpy(), L_grid.numpy(), levels=30, cmap="viridis")
plt.colorbar(contour, label="Loss")
ax.plot(w_history, b_history, "r.-", markersize=3, linewidth=1, label="GD path")
ax.scatter([2.0], [1.0], color="white", s=100, marker="*", zorder=5, label="Optimal")
ax.set_xlabel("w", fontsize=12)
ax.set_ylabel("b", fontsize=12)
ax.set_title("Loss Landscape & Gradient Descent Path", fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

---
## ✏️ Exercises

In [ ]:
# Exercise 1: Modify the training loop to learn y = 3x - 2 instead of y = 2x + 1.
# Generate new data with this ground truth and train a model.

# Your code here:

In [ ]:
# Exercise 2: Try learning rate = 0.5. What happens? Does the loss converge or explode?

# Your code here:

---
## 📝 Key Takeaways

1. **Gradient descent** = forward → loss → backward → update → zero_grad.
2. The **gradient** tells you which direction to move each weight to reduce loss.
3. **Learning rate** controls step size — too small is slow, too large can diverge.
4. The **loss landscape** is the surface we're trying to find the minimum of.
5. Gradient descent follows the steepest downhill path on this surface.

**Tomorrow:** We'll do a deep dive into backpropagation — the chain rule in action!